
1. Instalar librerías
---



In [ ]:
!pip install meteostat==1.6.8 --no-deps -q
!pip install openpyxl tqdm -q

2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


3. Importar librerías

In [ ]:
from pathlib import Path
from datetime import datetime, date, time, timedelta
import pandas as pd
import numpy as np
import re
from tqdm import tqdm

# Fix para Meteostat con NumPy moderno
if not hasattr(np, "NaN"):
    np.NaN = np.nan

from meteostat import Point, Hourly

tqdm.pandas()

print("Entorno cargado correctamente")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Entorno cargado correctamente
Pandas: 2.2.2
NumPy: 2.0.2


4. Definir rutas

In [ ]:
ruta_archivo = Path('/content/drive/MyDrive/Final_Esp_Data/Dataset_Clima.xlsx')
carpeta = Path('/content/drive/MyDrive/Final_Esp_Data')

ruta_salida_excel = carpeta / 'Dataset_Clima_Con_Historico.xlsx'
ruta_salida_csv = carpeta / 'Dataset_Clima_Con_Historico.csv'

print("Archivo existe:", ruta_archivo.exists())

Archivo existe: True


5. Cargar archivo

In [ ]:
df = pd.read_excel(ruta_archivo)
df.columns = df.columns.str.strip()

print("Filas:", len(df))
print("Columnas:", df.columns.tolist())

df.head()

Filas: 1645
Columnas: ['Longitud Sexa finca', 'Latitud Sexa finca', 'Municipio Finca', 'HORA INICIO TURNO', 'HORA FIN TURNO', 'FINCA', 'FECHA']


,Longitud Sexa finca,Latitud Sexa finca,Municipio Finca,HORA INICIO TURNO,HORA FIN TURNO,FINCA,FECHA
0,75°50'28.7''W,4°10'32.5''N,Sevilla,06:00:00,14:00:00,13INDOS,2026-01-01
1,75°50'28.7''W,4°10'32.5''N,Sevilla,14:00:00,22:00:00,13INDOS,2026-01-01
2,75°51'47.2''W,4°07'04.0''N,Sevilla,06:00:00,14:00:00,13CRIST,2026-01-01
3,75°51'47.2''W,4°07'04.0''N,Sevilla,14:00:00,22:00:00,13CRIST,2026-01-01
4,75°50'28.7''W,4°10'32.5''N,Sevilla,06:00:00,14:00:00,13INDOS,2026-01-02


6. Validar columnas requeridas

In [ ]:
columnas_requeridas = [
    "Longitud Sexa finca",
    "Latitud Sexa finca",
    "Municipio Finca",
    "HORA INICIO TURNO",
    "HORA FIN TURNO",
    "FECHA"
]

faltantes = [c for c in columnas_requeridas if c not in df.columns]

if faltantes:
    raise ValueError(f"Faltan columnas obligatorias: {faltantes}")

print("Columnas obligatorias completas")

Columnas obligatorias completas


7. Funciones de limpieza

In [ ]:
def dms_to_decimal(valor):
    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip().upper()
    texto = texto.replace("''", '"')
    texto = texto.replace("″", '"')
    texto = texto.replace("”", '"')
    texto = texto.replace("’", "'")
    texto = texto.replace("′", "'")
    texto = texto.replace(" ", "")

    patron = r"(\d+(?:\.\d+)?)°(\d+(?:\.\d+)?)'(\d+(?:\.\d+)?)\"?([NSEW])"
    match = re.search(patron, texto)

    if match:
        grados = float(match.group(1))
        minutos = float(match.group(2))
        segundos = float(match.group(3))
        direccion = match.group(4)

        decimal = grados + minutos / 60 + segundos / 3600

        if direccion in ["S", "W"]:
            decimal *= -1

        return decimal

    try:
        return float(texto.replace(",", "."))
    except:
        return np.nan


def extraer_hora(valor):
    if pd.isna(valor):
        return None

    if isinstance(valor, time):
        return valor

    if isinstance(valor, datetime):
        return valor.time()

    texto = str(valor).strip()

    try:
        return pd.to_datetime(texto, errors="coerce").time()
    except:
        return None


def construir_datetime(fecha_valor, hora_valor):
    fecha = pd.to_datetime(fecha_valor, errors="coerce")

    if pd.isna(fecha):
        return pd.NaT

    hora = extraer_hora(hora_valor)

    if hora is None:
        return pd.NaT

    return datetime.combine(fecha.date(), hora)


def clasificar_lluvia(mm):
    if pd.isna(mm):
        return "Sin dato"

    if mm > 0:
        return "Lluvioso"

    return "Seco"

8. Preparar coordenadas, fechas y horas

In [ ]:
df["lat_decimal"] = df["Latitud Sexa finca"].apply(dms_to_decimal)
df["lon_decimal"] = df["Longitud Sexa finca"].apply(dms_to_decimal)

df["fecha_dt"] = pd.to_datetime(df["FECHA"], errors="coerce")

df["hora_inicio_turno_dt"] = df.apply(
    lambda row: construir_datetime(row["FECHA"], row["HORA INICIO TURNO"]),
    axis=1
)

df["hora_fin_turno_dt"] = df.apply(
    lambda row: construir_datetime(row["FECHA"], row["HORA FIN TURNO"]),
    axis=1
)

# Si el turno termina al día siguiente
df.loc[
    df["hora_fin_turno_dt"] <= df["hora_inicio_turno_dt"],
    "hora_fin_turno_dt"
] = df["hora_fin_turno_dt"] + timedelta(days=1)

df[[
    "Longitud Sexa finca",
    "Latitud Sexa finca",
    "lat_decimal",
    "lon_decimal",
    "FECHA",
    "HORA INICIO TURNO",
    "hora_inicio_turno_dt",
    "HORA FIN TURNO",
    "hora_fin_turno_dt"
]].head()

,Longitud Sexa finca,Latitud Sexa finca,lat_decimal,lon_decimal,FECHA,HORA INICIO TURNO,hora_inicio_turno_dt,HORA FIN TURNO,hora_fin_turno_dt
0,75°50'28.7''W,4°10'32.5''N,4.175694,-75.841306,2026-01-01,06:00:00,2026-01-01 06:00:00,14:00:00,2026-01-01 14:00:00
1,75°50'28.7''W,4°10'32.5''N,4.175694,-75.841306,2026-01-01,14:00:00,2026-01-01 14:00:00,22:00:00,2026-01-01 22:00:00
2,75°51'47.2''W,4°07'04.0''N,4.117778,-75.863111,2026-01-01,06:00:00,2026-01-01 06:00:00,14:00:00,2026-01-01 14:00:00
3,75°51'47.2''W,4°07'04.0''N,4.117778,-75.863111,2026-01-01,14:00:00,2026-01-01 14:00:00,22:00:00,2026-01-01 22:00:00
4,75°50'28.7''W,4°10'32.5''N,4.175694,-75.841306,2026-01-02,06:00:00,2026-01-02 06:00:00,14:00:00,2026-01-02 14:00:00


9. Validación crítica

In [ ]:
print("Total registros:", len(df))
print("Sin latitud:", df["lat_decimal"].isna().sum())
print("Sin longitud:", df["lon_decimal"].isna().sum())
print("Sin fecha:", df["fecha_dt"].isna().sum())
print("Sin hora inicio:", df["hora_inicio_turno_dt"].isna().sum())
print("Sin hora fin:", df["hora_fin_turno_dt"].isna().sum())

Total registros: 1645
Sin latitud: 0
Sin longitud: 0
Sin fecha: 0
Sin hora inicio: 2
Sin hora fin: 2


10. Función optimizada para consultar Meteostat

In [ ]:
cache_clima = {}

def consultar_clima_turno(row):
    lat = row["lat_decimal"]
    lon = row["lon_decimal"]
    inicio = row["hora_inicio_turno_dt"]
    fin = row["hora_fin_turno_dt"]

    if pd.isna(lat) or pd.isna(lon) or pd.isna(inicio) or pd.isna(fin):
        return pd.Series({
            "clima_temp_inicio_c": np.nan,
            "clima_precipitacion_inicio_mm": np.nan,
            "clima_estado_inicio": "Sin dato",
            "clima_precipitacion_turno_mm": np.nan,
            "clima_estado_turno": "Sin dato",
            "clima_humedad_promedio": np.nan,
            "clima_viento_promedio_kmh": np.nan,
            "clima_presion_promedio_hpa": np.nan,
            "clima_horas_consultadas": 0,
            "clima_observacion": "Datos insuficientes"
        })

    inicio = inicio.replace(minute=0, second=0, microsecond=0)
    fin = fin.replace(minute=0, second=0, microsecond=0)

    if fin <= inicio:
        fin = inicio + timedelta(hours=1)

    key = (
        round(float(lat), 5),
        round(float(lon), 5),
        inicio.strftime("%Y-%m-%d %H:%M:%S"),
        fin.strftime("%Y-%m-%d %H:%M:%S")
    )

    if key in cache_clima:
        return cache_clima[key]

    try:
        punto = Point(float(lat), float(lon))
        data = Hourly(punto, inicio, fin).fetch()

        if data.empty:
            resultado = pd.Series({
                "clima_temp_inicio_c": np.nan,
                "clima_precipitacion_inicio_mm": np.nan,
                "clima_estado_inicio": "Sin dato",
                "clima_precipitacion_turno_mm": np.nan,
                "clima_estado_turno": "Sin dato",
                "clima_humedad_promedio": np.nan,
                "clima_viento_promedio_kmh": np.nan,
                "clima_presion_promedio_hpa": np.nan,
                "clima_horas_consultadas": 0,
                "clima_observacion": "Meteostat no encontró datos para esa fecha/hora"
            })

            cache_clima[key] = resultado
            return resultado

        primera_hora = data.iloc[0]

        temp_inicio = primera_hora.get("temp", np.nan)
        prcp_inicio = primera_hora.get("prcp", np.nan)

        prcp_turno = data["prcp"].sum(skipna=True) if "prcp" in data.columns else np.nan
        humedad_prom = data["rhum"].mean(skipna=True) if "rhum" in data.columns else np.nan
        viento_prom = data["wspd"].mean(skipna=True) if "wspd" in data.columns else np.nan
        presion_prom = data["pres"].mean(skipna=True) if "pres" in data.columns else np.nan

        resultado = pd.Series({
            "clima_temp_inicio_c": temp_inicio,
            "clima_precipitacion_inicio_mm": prcp_inicio,
            "clima_estado_inicio": clasificar_lluvia(prcp_inicio),
            "clima_precipitacion_turno_mm": prcp_turno,
            "clima_estado_turno": clasificar_lluvia(prcp_turno),
            "clima_humedad_promedio": humedad_prom,
            "clima_viento_promedio_kmh": viento_prom,
            "clima_presion_promedio_hpa": presion_prom,
            "clima_horas_consultadas": len(data),
            "clima_observacion": "Dato obtenido correctamente"
        })

        cache_clima[key] = resultado
        return resultado

    except Exception as e:
        resultado = pd.Series({
            "clima_temp_inicio_c": np.nan,
            "clima_precipitacion_inicio_mm": np.nan,
            "clima_estado_inicio": "Error",
            "clima_precipitacion_turno_mm": np.nan,
            "clima_estado_turno": "Error",
            "clima_humedad_promedio": np.nan,
            "clima_viento_promedio_kmh": np.nan,
            "clima_presion_promedio_hpa": np.nan,
            "clima_horas_consultadas": 0,
            "clima_observacion": str(e)
        })

        cache_clima[key] = resultado
        return resultado

11. Ejecutar consulta climática

In [ ]:
df_clima = df.progress_apply(consultar_clima_turno, axis=1)

df_final = pd.concat([df, df_clima], axis=1)

df_final["fue_lluvioso_en_inicio"] = df_final["clima_precipitacion_inicio_mm"].apply(
    lambda x: "Sí" if pd.notna(x) and x > 0 else ("No" if pd.notna(x) else "Sin dato")
)

df_final["fue_lluvioso_en_turno"] = df_final["clima_precipitacion_turno_mm"].apply(
    lambda x: "Sí" if pd.notna(x) and x > 0 else ("No" if pd.notna(x) else "Sin dato")
)

df_final.head()

100%|██████████| 1645/1645 [00:41<00:00, 39.55it/s]


,Longitud Sexa finca,Latitud Sexa finca,Municipio Finca,HORA INICIO TURNO,HORA FIN TURNO,FINCA,FECHA,lat_decimal,lon_decimal,fecha_dt,...,clima_estado_inicio,clima_precipitacion_turno_mm,clima_estado_turno,clima_humedad_promedio,clima_viento_promedio_kmh,clima_presion_promedio_hpa,clima_horas_consultadas,clima_observacion,fue_lluvioso_en_inicio,fue_lluvioso_en_turno
0,75°50'28.7''W,4°10'32.5''N,Sevilla,06:00:00,14:00:00,13INDOS,2026-01-01,4.175694,-75.841306,2026-01-01,...,Sin dato,NaN,Sin dato,NaN,NaN,NaN,0,Meteostat no encontró datos para esa fecha/hora,Sin dato,Sin dato
1,75°50'28.7''W,4°10'32.5''N,Sevilla,14:00:00,22:00:00,13INDOS,2026-01-01,4.175694,-75.841306,2026-01-01,...,Sin dato,NaN,Sin dato,NaN,NaN,NaN,0,Meteostat no encontró datos para esa fecha/hora,Sin dato,Sin dato
2,75°51'47.2''W,4°07'04.0''N,Sevilla,06:00:00,14:00:00,13CRIST,2026-01-01,4.117778,-75.863111,2026-01-01,...,Sin dato,NaN,Sin dato,NaN,NaN,NaN,0,Meteostat no encontró datos para esa fecha/hora,Sin dato,Sin dato
3,75°51'47.2''W,4°07'04.0''N,Sevilla,14:00:00,22:00:00,13CRIST,2026-01-01,4.117778,-75.863111,2026-01-01,...,Sin dato,NaN,Sin dato,NaN,NaN,NaN,0,Meteostat no encontró datos para esa fecha/hora,Sin dato,Sin dato
4,75°50'28.7''W,4°10'32.5''N,Sevilla,06:00:00,14:00:00,13INDOS,2026-01-02,4.175694,-75.841306,2026-01-02,...,Sin dato,NaN,Sin dato,NaN,NaN,NaN,0,Meteostat no encontró datos para esa fecha/hora,Sin dato,Sin dato


12. Revisar resultados

In [ ]:
print("Resumen clima hora inicio:")
print(df_final["clima_estado_inicio"].value_counts(dropna=False))

print("\nResumen clima durante turno:")
print(df_final["clima_estado_turno"].value_counts(dropna=False))

df_final[[
    "Municipio Finca",
    "FINCA" if "FINCA" in df_final.columns else "Municipio Finca",
    "FECHA",
    "HORA INICIO TURNO",
    "HORA FIN TURNO",
    "lat_decimal",
    "lon_decimal",
    "clima_temp_inicio_c",
    "clima_precipitacion_inicio_mm",
    "clima_estado_inicio",
    "clima_precipitacion_turno_mm",
    "clima_estado_turno",
    "fue_lluvioso_en_turno",
    "clima_observacion"
]].head(20)

Resumen clima hora inicio:
clima_estado_inicio
Sin dato    1400
Seco         156
Lluvioso      89
Name: count, dtype: int64

Resumen clima durante turno:
clima_estado_turno
Sin dato    1355
Lluvioso     199
Seco          91
Name: count, dtype: int64


,Municipio Finca,FINCA,FECHA,HORA INICIO TURNO,HORA FIN TURNO,lat_decimal,lon_decimal,clima_temp_inicio_c,clima_precipitacion_inicio_mm,clima_estado_inicio,clima_precipitacion_turno_mm,clima_estado_turno,fue_lluvioso_en_turno,clima_observacion
0,Sevilla,13INDOS,2026-01-01,06:00:00,14:00:00,4.175694,-75.841306,NaN,NaN,Sin dato,NaN,Sin dato,Sin dato,Meteostat no encontró datos para esa fecha/hora
1,Sevilla,13INDOS,2026-01-01,14:00:00,22:00:00,4.175694,-75.841306,NaN,NaN,Sin dato,NaN,Sin dato,Sin dato,Meteostat no encontró datos para esa fecha/hora
2,Sevilla,13CRIST,2026-01-01,06:00:00,14:00:00,4.117778,-75.863111,NaN,NaN,Sin dato,NaN,Sin dato,Sin dato,Meteostat no encontró datos para esa fecha/hora
3,Sevilla,13CRIST,2026-01-01,14:00:00,22:00:00,4.117778,-75.863111,NaN,NaN,Sin dato,NaN,Sin dato,Sin dato,Meteostat no encontró datos para esa fecha/hora
4,Sevilla,13INDOS,2026-01-02,06:00:00,14:00:00,4.175694,-75.841306,NaN,NaN,Sin dato,NaN,Sin dato,Sin dato,Meteostat no encontró datos para esa fecha/hora
5,Sevilla,13INDOS,2026-01-02,14:00:00,22:00:00,4.175694,-75.841306,NaN,NaN,Sin dato,NaN,Sin dato,Sin dato,Meteostat no encontró datos para esa fecha/hora
6,Sevilla,13CRIST,2026-01-02,06:00:00,14:00:00,4.117778,-75.863111,NaN,NaN,Sin dato,NaN,Sin dato,Sin dato,Meteostat no encontró datos para esa fecha/hora
7,Sevilla,13CRIST,2026-01-02,14:00:00,22:00:00,4.117778,-75.863111,NaN,NaN,Sin dato,NaN,Sin dato,Sin dato,Meteostat no encontró datos para esa fecha/hora
8,Bolivar,35PLAYA,2026-01-02,06:00:00,14:00:00,4.425917,-76.263222,NaN,NaN,Sin dato,NaN,Sin dato,Sin dato,Meteostat no encontró datos para esa fecha/hora
9,Bolivar,35PLAYA,2026-01-02,14:00:00,22:00:00,4.425917,-76.263222,NaN,NaN,Sin dato,NaN,Sin dato,Sin dato,Meteostat no encontró datos para esa fecha/hora


In [ ]:
total = len(df_final)
sin_data = df_final["clima_observacion"].str.contains("no encontró", case=False, na=False).sum()

print(f"Sin data: {sin_data} ({sin_data/total:.2%})")

Sin data: 1353 (82.25%)


13. Guardar archivo final

In [ ]:
df_final.to_excel(ruta_salida_excel, index=False)
df_final.to_csv(ruta_salida_csv, index=False, encoding="utf-8-sig")

print("Archivo Excel generado:")
print(ruta_salida_excel)

print("Archivo CSV generado:")
print(ruta_salida_csv)